### Project  Title: Customer Financial Credit & Risk Analysis Dashboard

In [188]:
import pandas as pd
import numpy as np


In [189]:
df = pd.read_excel("Customer_Financial_Credit_and_Risk_Analysis.xlsx")
df.head()

,Customer_ID,Age,Gender,Education_Level,Marital_Status,Number_of _Dependents,City,State,Country,Employment _Status,...,Loan_Amount,Interest_Rate,Assets_Value,Debt_to_Income_Ratio,Loan_Purpose,Loan_Status,Credit_Score,Previous Defaults,Payment_History,Risk_Rating
0,CUST0001,49,Male,PhD,Divorced,0.0,Mumbai,Maharashtra,India,Unemployed,...,45713.0,0.09,120228.0,0.154313,Business,Approved,688.0,2.0,Poor,Low
1,CUST0002,57,Female,Bachelor's,Widowed,0.0,Delhi,Delhi,India,Employed,...,33835.0,0.09,55849.0,0.148920,Auto,Rejected,690.0,3.0,Fair,Medium
2,CUST0003,21,Non-binary,Master's,Single,3.0,Bangalore,Karnataka,India,Employed,...,36623.0,0.12,180700.0,0.362398,Home,Rejected,600.0,3.0,Fair,Medium
3,CUST0004,59,Male,Bachelor's,Single,3.0,Chennai,Tamil Nadu,India,Unemployed,...,26541.0,0.12,157319.0,0.454964,Personal,Rejected,622.0,4.0,Excellent,Medium
4,CUST0005,25,Non-binary,Bachelor's,Widowed,NaN,Hyderabad,Telangana,India,Unemployed,...,36528.0,0.06,287140.0,0.143242,Personal,Approved,766.0,3.0,Fair,Low


In [190]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
)



In [191]:
df.columns


Index(['Customer_ID', 'Age', 'Gender', 'Education_Level', 'Marital_Status',
       'Number_of_Dependents', 'City', 'State', 'Country', 'Employment_Status',
       'Years_at_Current_Job', 'Income', 'Loan_Amount', 'Interest_Rate',
       'Assets_Value', 'Debt_to_Income_Ratio', 'Loan_Purpose', 'Loan_Status',
       'Credit_Score', 'Previous_Defaults', 'Payment_History', 'Risk_Rating'],
      dtype='str')

In [192]:
df['Interest_Rate'] = df['Interest_Rate'].astype(str).str.replace('%','').astype(float)

## HANDLE MISSING VALUES

In [193]:
# Numeric columns
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns (SAFE FIX)
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [194]:
# TYPE FIXING
df['Number_of_Dependents'] = df['Number_of_Dependents'].fillna(0).astype(int)
df['Previous_Defaults'] = df['Previous_Defaults'].fillna(0).astype(int)

## FEATURE ENGINEERING 

In [195]:
# Loan to Income
df['Loan_to_Income'] = np.where(
    df['Income'] > 0,
    df['Loan_Amount'] / df['Income'],
    0
)

# Income Group (PAGE 3)
df['Income_Group'] = pd.cut(
    df['Income'],
    bins=[0,30000,70000,150000,1000000],
    labels=['Low','Medium','High','Very High']
)

# Loan Group
df['Loan_Group'] = pd.cut(
    df['Loan_Amount'],
    bins=[0,5000,15000,30000,100000],
    labels=['Small','Medium','Large','Very Large']
)

# Credit Group
df['Credit_Group'] = pd.cut(
    df['Credit_Score'],
    bins=[300,580,670,740,850],
    labels=['Poor','Fair','Good','Excellent']
)

# Age Group (PAGE 1)
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[17,30,45,60,100],
    labels=['Young','Adult','Mid-Age','Senior']
)


In [196]:
df['Risk_Score'] = (
    df['Debt_to_Income_Ratio'] * 30 +
    (850 - df['Credit_Score']) / 10 +
    df['Previous_Defaults'] * 15 +
    df['Payment_History'].map({
        'Poor':20,
        'Fair':10,
        'Good':5,
        'Excellent':0
    }).fillna(0)
)

In [197]:
df['Risk_Rating'] = pd.cut(
    df['Risk_Score'],
    bins=[0, 40, 75, np.inf],
    labels=['Low', 'Medium', 'High']
)

In [198]:
print(df['Risk_Rating'].value_counts())

Risk_Rating
Medium    4255
High      2548
Low       1219
Name: count, dtype: int64


In [199]:
df['High_Risk_Flag'] = np.where(
    (
        (df['Credit_Score'] < 620).astype(int) +
        (df['Debt_to_Income_Ratio'] > 0.45).astype(int) +
        (df['Previous_Defaults'] > 0).astype(int) +
        (df['Payment_History'] == 'Poor').astype(int)
    ) >= 2,
    1, 0
)

In [200]:
df['Default_Flag'] = np.where(
    (df['Previous_Defaults'] >= 2) |
    (df['Credit_Score'] < 600),
    1, 0
)

In [201]:
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Shape: (8022, 30)
Columns: 30


In [202]:
missing = df.isnull().sum().sum()
print("Total Missing Values:", missing)

Total Missing Values: 0


In [203]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [204]:
print(df.dtypes)


Customer_ID                  str
Age                        int64
Gender                       str
Education_Level              str
Marital_Status               str
Number_of_Dependents       int64
City                         str
State                        str
Country                      str
Employment_Status            str
Years_at_Current_Job       int64
Income                   float64
Loan_Amount              float64
Interest_Rate            float64
Assets_Value             float64
Debt_to_Income_Ratio     float64
Loan_Purpose                 str
Loan_Status                  str
Credit_Score             float64
Previous_Defaults          int64
Payment_History              str
Risk_Rating             category
Loan_to_Income           float64
Income_Group            category
Loan_Group              category
Credit_Group            category
Age_Group               category
Risk_Score               float64
High_Risk_Flag             int64
Default_Flag               int64
dtype: obj

In [205]:
category_cols = [
    'Risk_Rating',
    'Income_Group',
    'Loan_Group',
    'Credit_Group',
    'Age_Group'
]

for col in category_cols:
    df[col] = df[col].astype(str)

In [206]:
print(df.dtypes)

Customer_ID                 str
Age                       int64
Gender                      str
Education_Level             str
Marital_Status              str
Number_of_Dependents      int64
City                        str
State                       str
Country                     str
Employment_Status           str
Years_at_Current_Job      int64
Income                  float64
Loan_Amount             float64
Interest_Rate           float64
Assets_Value            float64
Debt_to_Income_Ratio    float64
Loan_Purpose                str
Loan_Status                 str
Credit_Score            float64
Previous_Defaults         int64
Payment_History             str
Risk_Rating                 str
Loan_to_Income          float64
Income_Group                str
Loan_Group                  str
Credit_Group                str
Age_Group                   str
Risk_Score              float64
High_Risk_Flag            int64
Default_Flag              int64
dtype: object


In [207]:
print(df['Loan_Status'].value_counts(normalize=True) * 100)

Loan_Status
Rejected    51.13438
Approved    48.86562
Name: proportion, dtype: float64


In [208]:
print(df['Risk_Rating'].value_counts(normalize=True) * 100)

Risk_Rating
Medium    53.041636
High      31.762653
Low       15.195712
Name: proportion, dtype: float64


In [209]:
print(df['Risk_Score'].describe())

count    8022.000000
mean       64.046214
std        21.848047
min         9.620960
25%        47.962115
50%        64.026509
75%        79.670381
max       122.638005
Name: Risk_Score, dtype: float64


In [210]:
print(df.groupby('Risk_Rating')['Credit_Score'].mean())

Risk_Rating
High      686.366954
Low       719.092699
Medium    701.090482
Name: Credit_Score, dtype: float64


In [211]:
print("Total Customers:", df['Customer_ID'].nunique())
print("Total Loans:", len(df))
print("Total Loan Amount:", df['Loan_Amount'].sum())
print("Approval Rate %:", df['Loan_Status'].eq('Approved').mean() * 100)


Total Customers: 8022
Total Loans: 8022
Total Loan Amount: 220734458.0
Approval Rate %: 48.86561954624782


In [212]:
# PAGE 1 KPIs (BUSINESS VIEW)

total_customers = df['Customer_ID'].nunique()
total_loans = len(df)
total_loan_amount = df['Loan_Amount'].sum()

approved_loans = (df['Loan_Status'] == 'Approved').sum()
rejected_loans = (df['Loan_Status'] == 'Rejected').sum()

approval_rate = (df['Loan_Status'].eq('Approved').mean()) * 100

print("Total Customers:", total_customers)
print("Total Loans:", total_loans)
print("Total Loan Amount:", total_loan_amount)
print("Approved Loans:", approved_loans)
print("Rejected Loans:", rejected_loans)
print("Approval Rate %:", approval_rate)

Total Customers: 8022
Total Loans: 8022
Total Loan Amount: 220734458.0
Approved Loans: 3920
Rejected Loans: 4102
Approval Rate %: 48.86561954624782


In [213]:
# High Risk Customers (model-based)
high_risk_customers = df['High_Risk_Flag'].sum()

# Serious default rate (>= 2 defaults)
serious_default_rate = (df['Previous_Defaults'] >= 2).mean() * 100

# Average Credit Score
avg_credit_score = df['Credit_Score'].mean()

# Average Debt-to-Income Ratio
avg_dti = df['Debt_to_Income_Ratio'].mean()


# OUTPUT
print("High Risk Customers:", high_risk_customers)
print("Serious Default Rate %:", serious_default_rate)
print("Average Credit Score:", avg_credit_score)
print("Average DTI:", avg_dti)

High Risk Customers: 3584
Serious Default Rate %: 65.79406631762653
Average Credit Score: 699.1494639740713
Average DTI: 0.3495447208682204


In [214]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape: (8022, 30)
Missing values: 0
Duplicate rows: 0


## 11. FINAL EXPORT 


In [215]:
df.to_csv("financial_credit_risk_cleaned_data.csv", index=False)


In [217]:
summery = df.describe(include='all')
summery.to_csv("credit_risk_dataset_summary.csv")